## SCD Type 2


In [0]:
import os
from pyspark.sql import functions as f
from pyspark.sql.types import StructType

In [0]:
df_customers = spark.sql("""
                         select * from 
                         parquet.`/Volumes/workspace/default/medallion_data/output/bronze/customers/`
                         """)

In [0]:
df_customers=df_customers.dropDuplicates().dropna()

In [0]:
from pyspark.sql.functions import concat_ws, md5

def generate_surrogate_key(df, key_column_name, source_columns):
    """Generate a surrogate key by hashing the source columns."""
    return df.withColumn(
        key_column_name,
        md5(concat_ws("|", *[f.col(c) for c in source_columns]))
    )

def process_customers(spark):
    bronze_path = "/Volumes/workspace/default/medallion_data/output/bronze/customers/"
    silver_path = "/Volumes/workspace/default/medallion_data/output/silver/customers"
    
    # 1. Read incoming Bronze data & Clean
    df_bronze = spark.read.parquet(bronze_path)
    df_incoming = df_bronze.withColumn("name", f.initcap(f.col("name")))
    
    # 2. Read or Initialize Existing Silver Table
    # If Day 1, Silver doesn't exist yet, so we create an empty DataFrame with the expected schema
    if os.path.exists(silver_path):
        df_existing_silver = spark.read.parquet(silver_path)
    else:
        # Create empty schema matching your target structure
        schema = df_incoming.schema \
            .add("customer_sk", "string") \
            .add("start_date", "date") \
            .add("end_date", "date") \
            .add("is_active", "boolean")
        df_existing_silver = spark.createDataFrame([], schema)

    # 3. Separate Active and Inactive Records from existing Silver
    df_silver_active = df_existing_silver.filter(f.col("is_active") == True)
    df_silver_inactive = df_existing_silver.filter(f.col("is_active") == False)

    # 4. Identify Records that have actual changes in City or Tier
    # Join active records with incoming records on ID
    df_joined = df_silver_active.alias("s").join(
        df_incoming.alias("b"), 
        f.col("s.customer_id") == f.col("b.customer_id"), 
        "inner"
    )
    
    # Filter for rows where attributes changed
    df_changed_records = df_joined.filter(
        (f.col("s.city") != f.col("b.city")) | 
        (f.col("s.tier") != f.col("b.tier"))
    )

    # 5. EXPIRE OLD RECORDS: Update changed records to inactive
    # (Using the event_ts of the incoming record as the end_date)
    df_expired = df_changed_records.select("s.*", f.col("b.event_ts").alias("new_event_ts")) \
        .withColumn("is_active", f.lit(False)) \
        .withColumn("end_date", f.to_date(f.col("new_event_ts"))) \
        .drop("new_event_ts")

    # 6. PRESERVE UNCHANGED RECORDS
    # Records in silver that did NOT experience a change
    df_unchanged = df_silver_active.join(
        df_changed_records.select("s.customer_id"), 
        "customer_id", 
        "left_anti"
    )

    # 7. PREPARE NEW & UPDATED INCOMING RECORDS FOR INSERTION
    # Generate Surrogate Key for incoming rows
    df_new_inserts = generate_surrogate_key(df_incoming, "customer_sk", ["customer_id", "event_ts"])
    
    # Standardize tracking metadata columns
    df_new_inserts = df_new_inserts \
        .withColumn("start_date", f.to_date(f.col("event_ts"))) \
        .withColumn("end_date", f.lit(None).cast("date")) \
        .withColumn("is_active", f.lit(True))

    # 8. UNION EVERYTHING TOGETHER
    # Combine (Unchanged Active + Expired Old + New Inserts + Historical Inactive)
    df_final_silver = df_unchanged.unionByName(df_expired) \
                                  .unionByName(df_new_inserts) \
                                  .unionByName(df_silver_inactive)

    # 9. Write back out to Silver (Overwrite current snapshot)
    df_final_silver.write.mode("overwrite").parquet(silver_path)
    print("Silver Customer Dimension (SCD2) processed successfully.")

In [0]:
process_customers(spark)